## Cell 1: Import Libraries

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

## Cell 2: Load Data

In [3]:
stock_returns = pd.read_csv(
    "../data/stock_returns_log.csv", index_col=0, parse_dates=True
)
etf_returns = pd.read_csv("../data/etf_returns_log.csv", index_col=0, parse_dates=True)
stock_meta = pd.read_csv("../data/stock_meta.csv", index_col=0)

print(f"Stock returns: {stock_returns.shape}")
print(f"ETF returns:   {etf_returns.shape}")
print(f"Stock meta:    {stock_meta.shape}")

Stock returns: (1508, 410)
ETF returns:   (1508, 15)
Stock meta:    (505, 3)


## Cell 3: Set Parameters

In [4]:
# Number of eigenportfolios to retain as risk factors
N_FACTORS = 15
# 252 trading days = approximately one calendar year
PCA_WINDOW = 252

## Cell 4: Create Correlation Matrix

In [ ]:
def run_pca(returns, n_factors=N_FACTORS):
    """
    Standardize returns and extract PCA factors.

    Parameters
    ----------
    returns_window : pd.DataFrame, shape (M, N)
        Raw log returns for N stocks over M days
    n_factors : int
        Number of principal components to extract

    Returns
    -------
    factors : np.array, shape (M, n_factors)
        Return series for each eigenportfolio (the risk factors)
    components : np.array, shape (n_factors, N)
        Eigenvectors — each row is one principal component
    explained_variance_ratio : np.array, shape (n_factors,)
        Fraction of total variance explained by each factor
    pca : fitted PCA object
        Kept so we can transform new data later
    scaler : fitted StandardScaler object
        Kept so we can standardize new data consistently
    """
    # Step 1: Standardize returns
    scaler = StandardScaler()
    raw_returns = returns.values  # Shape (M, N)
    scaled_returns = scaler.fit_transform(raw_returns)  # Shape (M, N)

    # Step 2: Run PCA
    pca = PCA(n_components=n_factors)
    factors = pca.fit(scaled_returns)

    # Step 3: Scale components by volatility
    returns_std = returns.std(ddof=1).values

    weights = pca.components_.T / returns_std.reshape(-1, 1)

    # Step 4: Compute Eigenportfolio Returns
    factors = raw_returns @ weights

    return factors, pca.components_, weights, pca.explained_variance_ratio_, pca, scaler
